# ASMR: Memory Operations

This notebook demonstrates basic memory operations in ASMR:
- Adding memories
- Updating memories (versioning)
- Searching memories
- Memory history and supersession

In [ ]:
# Setup path for local imports
import sys
sys.path.insert(0, '..')

from datetime import datetime, timedelta
from memory.store import MemoryStore
from memory.indexer import FAISSIndexer
from memory.temporal import TemporalManager

## 1. Initialize Memory Store

In [ ]:
# Create a simple embedding function for demo
def mock_embedding(text: str) -> list[float]:
    """Generate a deterministic mock embedding based on text hash."""
    import hashlib
    h = hashlib.md5(text.encode()).hexdigest()
    return [int(h[i:i+2], 16) / 255.0 for i in range(0, 32, 2)] * 24  # 384-dim

# Initialize store
store = MemoryStore(
    indexer=FAISSIndexer(dimension=384),
    embedding_fn=mock_embedding,
    supersession_threshold=0.9,
)

print("Memory store initialized")

## 2. Adding Memories

In [ ]:
# Add some memories
mem1 = store.add(
    content="The CEO of TechCorp is John Smith.",
    source="company_info",
    tags=["leadership", "company"],
)
print(f"Added memory: {mem1.id}")
print(f"  Content: {mem1.content}")
print(f"  Version: {mem1.version}")
print(f"  Active: {mem1.is_active}")

In [ ]:
# Add more memories
mem2 = store.add(
    content="TechCorp was founded in 2010 in Silicon Valley.",
    source="company_info",
    tags=["history", "company"],
)

mem3 = store.add(
    content="TechCorp's main product is enterprise software.",
    source="products",
    tags=["products", "company"],
)

print(f"Total memories: {len(store)}")

## 3. Updating Memories (Versioning)

In [ ]:
# Update the CEO information
mem1_updated = store.update(
    memory_id=mem1.id,
    new_content="The CEO of TechCorp is Jane Doe (appointed 2024).",
)

print(f"Updated memory: {mem1_updated.id}")
print(f"  Content: {mem1_updated.content}")
print(f"  Version: {mem1_updated.version}")
print(f"  Supersedes: {mem1_updated.supersedes}")

# Check old memory is now inactive
old_mem = store.get(mem1.id)
print(f"\nOld memory active: {old_mem.is_active}")

## 4. Memory History

In [ ]:
# Get version history
history = store.get_history(mem1_updated.id)

print("Version History:")
for mem in history:
    print(f"  v{mem.version}: {mem.content[:50]}... (active: {mem.is_active})")

## 5. Searching Memories

In [ ]:
# Search for memories (only returns active)
query = "Who is the CEO?"
query_embedding = mock_embedding(query)

results = store.search(query_embedding, top_k=3)

print(f"Search results for: '{query}'")
for mem in results:
    print(f"  - {mem.content}")
    print(f"    Source: {mem.source}, Active: {mem.is_active}")

## 6. Temporal Manager

In [ ]:
# Initialize temporal manager
temporal = TemporalManager(half_life_days=30)

# Calculate recency scores
now = datetime.utcnow()

timestamps = [
    now,  # Today
    now - timedelta(days=7),   # 1 week ago
    now - timedelta(days=30),  # 1 month ago
    now - timedelta(days=90),  # 3 months ago
]

print("Recency scores (half-life = 30 days):")
for ts in timestamps:
    score = temporal.recency_score(ts)
    days_ago = (now - ts).days
    print(f"  {days_ago} days ago: {score:.3f}")

In [ ]:
# Detect temporal language
texts = [
    "As of January 2024, the policy is updated.",
    "This was the old approach we used to follow.",
    "Currently, we recommend using the new method.",
]

print("Temporal language detection:")
for text in texts:
    detections = temporal.detect_temporal_language(text)
    print(f"  '{text[:40]}...'")
    for d in detections:
        print(f"    - {d['type']}: current={d.get('indicates_current')}, outdated={d.get('indicates_outdated')}")